# Train ACT on AlohaTransferCube

This notebook is the Colab version of `train.py` and `evaluate.py` in the mimic-arm repo.

You will:

1. Install LeRobot 0.6.1 with the ALOHA simulator, without throwing away Colab's CUDA PyTorch.
2. Train an ACT policy on `lerobot/aloha_sim_transfer_cube_human` for 8,000 steps.
3. Evaluate it for 20 episodes in `AlohaTransferCube` and watch one of the saved videos.

Use a **GPU runtime** (Runtime → Change runtime type → T4). A CPU runtime will train, but 8,000 steps will not finish during a free session.


## Why 8,000 steps, not 100,000

LeRobot's default ACT run is **100,000 steps** at batch size 8. The checkpoint Hugging Face published for this task (`lerobot/act_aloha_sim_transfer_cube_human`) was saved at **80,000** steps and is the one that actually hands the cube over.

A free T4 session often ends after a couple of hours, and sometimes sooner. 100,000 steps is an overnight job on a better GPU, not a free-tier notebook.

**8,000 steps** is the compromise used here:

- long enough that the loss should fall (the policy starts to imitate the demonstrations)
- short enough that a T4 can plausibly finish, including the dataset download
- checkpointed every 2,000 steps, so a disconnect keeps a usable folder

Expect a low success rate afterwards. The arms may start reaching. They will not reliably transfer the cube. If the session is still healthy at the end, train longer: resume from `outputs/train/act_aloha_transfer_cube_colab` and raise `--steps`. The loss curve is the signal to watch at this length, not the success rate.

On a CPU smoke-test machine, batch size 2 took about one second per step. A T4 at batch size 8 is much faster per sample, but 8,000 steps plus evaluation is still the bulk of a free session. Plan on roughly one to two hours. That number was not timed on a T4.


In [ ]:
import sys
import torch

print("Python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU", torch.cuda.get_device_name(0))
else:
    print("No GPU visible. Switch the runtime to a T4 before training 8000 steps.")

if sys.version_info < (3, 12):
    raise SystemExit(
        "LeRobot 0.6.1 needs Python 3.12 or newer. "
        "This Colab runtime is too old for the pinned install."
    )


## Install

Colab already has a CUDA build of PyTorch matched to its driver. `pip install lerobot` will try to replace it with `torch==2.11.0+cu130`, which a free T4's driver often cannot load. The cell below remembers Colab's torch, installs LeRobot, and puts the original torch back if CUDA disappeared.

System packages: EGL for MuJoCo-on-GPU, OSMesa as a CPU fallback.


In [ ]:
import importlib
import os
import subprocess
import sys

import torch
import torchvision

orig_torch = torch.__version__
orig_vision = torchvision.__version__
print("Keeping Colab torch", orig_torch, "vision", orig_vision)

subprocess.check_call(["apt-get", "update", "-qq"])
subprocess.check_call(
    [
        "apt-get", "install", "-y", "-qq",
        "libegl1", "libgl1", "libglib2.0-0", "libosmesa6", "ffmpeg",
    ]
)

# LeRobot 0.6.1 + dataset/training extras + gym-aloha + the eval video writer.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "lerobot[training,aloha,evaluation]==0.6.1",
        "numpy==2.2.6",
        "fsspec==2026.2.0",
    ]
)

importlib.invalidate_caches()
# Re-import after the install. A replaced torch module may still be the old one
# until the process restarts, so also ask a fresh interpreter.
probe = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.cuda.is_available())"],
    check=True,
    capture_output=True,
    text=True,
)
print(probe.stdout)
cuda_ok = probe.stdout.strip().splitlines()[-1] == "True"
if not cuda_ok and "+" in orig_torch:
    cuda_tag = orig_torch.split("+", 1)[1]
    if cuda_tag.startswith("cu"):
        index = f"https://download.pytorch.org/whl/{cuda_tag}"
        print(f"CUDA torch was replaced. Restoring {orig_torch} from {index}")
        subprocess.check_call(
            [
                sys.executable, "-m", "pip", "install", "--force-reinstall",
                f"torch=={orig_torch}",
                f"torchvision=={orig_vision}",
                "--index-url", index,
            ]
        )
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "numpy==2.2.6", "fsspec==2026.2.0"]
        )

import lerobot
print("lerobot", lerobot.__version__)


## Get the training scripts

`train.py` and `evaluate.py` are the same files you run locally. Colab only downloads the notebook, so this cell clones https://github.com/Barbodmz/mimic-arm if those files are not already here.


In [ ]:
import os
from pathlib import Path

if not Path("train.py").is_file():
    repo = os.environ.get("MIMIC_ARM_REPO", "https://github.com/Barbodmz/mimic-arm")
    __import__("subprocess").check_call(["git", "clone", "--depth", "1", repo, "mimic-arm"])
    os.chdir("mimic-arm")

print("Working directory:", Path.cwd())
print("train.py", Path("train.py").is_file(), "evaluate.py", Path("evaluate.py").is_file())


## Train

This is `python train.py` with the knobs that matter on a free T4:

- `--steps 8000` instead of the default 100,000
- `--batch-size 8`, which is LeRobot's ACT default and fits a T4 for this one-camera task. If the runtime dies with a CUDA out-of-memory error, change it to 4
- `--save_freq=2000` so you can resume
- `--env_eval_freq=0` so training does not spend its time rolling out 50 episodes in the simulator. The next cell does a 20-episode eval on purpose
- `--num-workers 2` to leave RAM for the GPU copy of the batch

The loss log every 200 steps is the thing to read. It should trend down. It will not reach zero.


In [ ]:
import subprocess
import sys

# Same flags as a local `python train.py ...` call. Anything train.py does not
# recognize is forwarded to lerobot-train.
subprocess.check_call(
    [
        sys.executable, "train.py",
        "--steps", "8000",
        "--batch-size", "8",
        "--device", "cuda",
        "--num-workers", "2",
        "--output-dir", "outputs/train/act_aloha_transfer_cube_colab",
        "--save_freq=2000",
        "--env_eval_freq=0",
        "--log_freq=200",
    ]
)


## Evaluate 20 episodes

`evaluate.py` loads `checkpoints/last/pretrained_model`, runs 20 episodes of `AlohaTransferCube`, prints the success rate and average reward, and saves two mp4s.

Success means the left gripper ended an episode holding the cube off the table (reward 4). After 8,000 steps, a success rate near zero is normal.


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable, "evaluate.py",
        "--checkpoint", "outputs/train/act_aloha_transfer_cube_colab/checkpoints/last/pretrained_model",
        "--episodes", "20",
        "--videos", "2",
        "--device", "cuda",
        "--output-dir", "outputs/eval/act_aloha_transfer_cube_colab",
    ]
)


## Watch a rollout

The video is one of the episodes just scored. A random policy mostly waves the arms. A trained one reaches for the red cube and tries to pass it to the other gripper.


In [ ]:
from pathlib import Path
from IPython.display import Video, display

videos = sorted(Path("outputs/eval/act_aloha_transfer_cube_colab").rglob("*.mp4"))
if not videos:
    raise SystemExit("No mp4 found. Check the evaluate.py log above.")
print(videos[0])
display(Video(str(videos[0]), embed=True))


## If the session dies

Resume from the last checkpoint. LeRobot stores the step inside it, and `train.py --resume` points `--config_path` at that folder:

```bash
python train.py --resume \
    --output-dir outputs/train/act_aloha_transfer_cube_colab \
    --steps 20000 \
    --device cuda
```

CLI flags override the saved config, so `--steps 20000` extends the run.

What to read next is the "Next steps" section of the README: recording your own simulated demonstrations, then the same ACT loop on a real SO-101 arm.
